# SayurKita — Notebook 3  INGREDIENTS MASTER





---
##   Ingredients Master + Nutrition

### 1. import dan load data

In [ ]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

df_master = pd.read_csv(
    '/content/ingredients_master.csv'
)
df_nutr = pd.read_csv(
    '/content/nutrition_cleaned.csv'
)

print(f"\n Master    : {len(df_master):>5} baris  |  kolom: {df_master.columns.tolist()}")
print(f" Nutrition : {len(df_nutr):>5} baris  |  kolom: {df_nutr.columns.tolist()}")



### 2. normalisasi teks

In [ ]:
def normalize(text: str) -> str:
    """Lowercase, strip, hapus karakter non-alfanumerik berlebih."""
    t = str(text).lower().strip()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_master['_key'] = df_master['nama_id'].apply(normalize)
df_nutr['_key_nutr'] = df_nutr['nama_bersih'].apply(normalize)

# Dict lookup: key_nutr → row index di df_nutr
nutr_lookup = dict(zip(df_nutr['_key_nutr'], df_nutr.index))
nutr_keys   = list(df_nutr['_key_nutr'])

### 3.  match

In [ ]:
FUZZY_THRESHOLD = 82

def get_nutrition(master_key: str) -> tuple:
    """
    Cari nutrisi untuk satu bahan.
    Return (kalori, protein, lemak, karbo, match_type) atau None.
    """
    # ── Layer 1: Exact match
    if master_key in nutr_lookup:
        row = df_nutr.iloc[nutr_lookup[master_key]]
        return (row['kalori_per_100g'], row['protein_g'],
                row['lemak_g'], row['karbo_g'], 'exact')

    # ── Layer 2: Token match
    # Ambil kata terpanjang dari nama bahan (kata inti, bukan preposisi)
    tokens = [t for t in master_key.split() if len(t) > 3]
    # Coba dari token terpanjang ke pendek
    tokens_sorted = sorted(tokens, key=len, reverse=True)
    for token in tokens_sorted[:3]:   # cek maks 3 token teratas
        if token in nutr_lookup:
            row = df_nutr.iloc[nutr_lookup[token]]
            return (row['kalori_per_100g'], row['protein_g'],
                    row['lemak_g'], row['karbo_g'], f'token:{token}')

    # ── Layer 3: Fuzzy match
    result = process.extractOne(
        master_key,
        nutr_keys,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_THRESHOLD
    )
    if result:
        matched_key, score, _ = result
        row = df_nutr.iloc[nutr_lookup[matched_key]]
        return (row['kalori_per_100g'], row['protein_g'],
                row['lemak_g'], row['karbo_g'], f'fuzzy:{score:.0f}%:{matched_key}')

    return None   # tidak ditemukan

### 4. merge

In [ ]:
print("\n⚙️  Menjalankan merge 3-lapis...")

kalori_list, protein_list, lemak_list, karbo_list, match_list = [], [], [], [], []

exact_count   = 0
token_count   = 0
fuzzy_count   = 0
nomatch_count = 0

for _, row in df_master.iterrows():
    result = get_nutrition(row['_key'])

    if result:
        kalori, protein, lemak, karbo, match_type = result
        kalori_list.append(kalori)
        protein_list.append(protein)
        lemak_list.append(lemak)
        karbo_list.append(karbo)
        match_list.append(match_type)

        if match_type == 'exact':
            exact_count += 1
        elif match_type.startswith('token'):
            token_count += 1
        else:
            fuzzy_count += 1
    else:
        kalori_list.append(0.0)
        protein_list.append(0.0)
        lemak_list.append(0.0)
        karbo_list.append(0.0)
        match_list.append('no_match')
        nomatch_count += 1

# Tambahkan kolom hasil ke master
df_master['kalori_per_100g'] = kalori_list
df_master['protein_g']       = protein_list
df_master['lemak_g']         = lemak_list
df_master['karbo_g']         = karbo_list
df_master['_match_type']     = match_list

### 5. statistik

In [ ]:
total = len(df_master)
matched = exact_count + token_count + fuzzy_count

print(f"\n HASIL MERGE ({total} bahan total):")
print(f"   Layer 1 — Exact match   : {exact_count:>5}  ({exact_count/total*100:.1f}%)")
print(f"   Layer 2 — Token match   : {token_count:>5}  ({token_count/total*100:.1f}%)")
print(f"   Layer 3 — Fuzzy match   : {fuzzy_count:>5}  ({fuzzy_count/total*100:.1f}%)")
print(f"   Tidak match (→ 0)       : {nomatch_count:>5}  ({nomatch_count/total*100:.1f}%)")

print(f"   Total coverage nutrisi  : {matched:>5}  ({matched/total*100:.1f}%)")

# Distribusi coverage per kategori
print(f"\n COVERAGE PER KATEGORI:")
cat_stats = df_master.groupby('kategori').apply(
    lambda g: pd.Series({
        'total': len(g),
        'dengan_nutrisi': (g['_match_type'] != 'no_match').sum(),
    })
).reset_index()
cat_stats['pct'] = (cat_stats['dengan_nutrisi'] / cat_stats['total'] * 100).round(1)
cat_stats = cat_stats.sort_values('pct', ascending=False)
for _, r in cat_stats.iterrows():
    bar = '█' * int(r['pct'] // 5)
    print(f"   {r['kategori']:<20}: {r['dengan_nutrisi']:>4}/{r['total']:<4} ({r['pct']:>5.1f}%) {bar}")

# Bahan yang tidak dapat nutrisi (no_match) — khusus yang frekuensi tinggi
no_match_df = df_master[df_master['_match_type'] == 'no_match'].sort_values(
    'frekuensi', ascending=False
)
print(f"\n  Top-20 bahan tanpa data nutrisi (NaN → 0, frekuensi tertinggi):")
print(f"   {'nama_id':<35} {'frekuensi':>10} {'kategori':<15}")
print(f"   {'-'*62}")
for _, r in no_match_df.head(20).iterrows():
    print(f"   {r['nama_id']:<35} {r['frekuensi']:>10} {r['kategori']:<15}")



for col in ['kalori_per_100g', 'protein_g', 'lemak_g', 'karbo_g']:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce').fillna(0.0)


for col in ['kalori_per_100g', 'protein_g', 'lemak_g', 'karbo_g']:
    df_master[col] = df_master[col].round(1)


FINAL_COLS = [
    'nama_id', 'frekuensi', 'kategori',
    'umur_kulkas', 'umur_suhu_ruang', 'umur_freezer',
    'kalori_per_100g', 'protein_g', 'lemak_g', 'karbo_g'
]
df_final = df_master[FINAL_COLS].copy()

### 6. preview 30 baris pertama

In [ ]:
print(f"\n{'='*100}")
print("PREVIEW — 30 BAHAN TERATAS (berdasarkan frekuensi resep)")
print(f"{'='*100}")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.1f}'.format)
print(df_final.head(30).to_string(index=False))

# Statistik
has_nutr = df_final[df_final['kalori_per_100g'] > 0]
print(f"\n📐 STATISTIK NILAI GIZI (dari {len(has_nutr)} bahan yang punya data nutrisi):")
print(has_nutr[['kalori_per_100g','protein_g','lemak_g','karbo_g']].describe().round(1).to_string())

### 7. export

In [ ]:
output_path = '/content/ingredients_master_nutrition.csv'
df_final.to_csv(output_path, index=False)
print(f" FILE TERSIMPAN: {output_path}")

---
##   Ingredients Master + Nutrition + Carbon

### import dan load data

In [ ]:
!pip install rapidfuzz -q

import json, re
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz


# 1. LOAD DATA

df = pd.read_csv("/content/ingredients_master_nutrition.csv")

with open("/content/data_karbon_sayurkita_id.json", "r", encoding="utf-8") as f:
    carbon_raw = json.load(f)

print(f" Ingredients master : {len(df)} bahan")
print(f" Carbon JSON        : {len(carbon_raw)} entri")
print(f"   Kolom CSV         : {df.columns.tolist()}")

### 2. normalisasi

In [ ]:
def norm_key(text: str) -> str:
    return text.replace("_", " ").lower().strip()

carbon = {}  # normalized_key → (original_key, co2e)
for orig_key, val in carbon_raw.items():
    carbon[norm_key(orig_key)] = (orig_key, val["co2e_per_kg"])

carbon_keys = list(carbon.keys())

### 3. MANUAL MAP —  untuk nama yang berbeda struktur

In [ ]:
MANUAL_MAP = {

    # ── PROTEIN HEWANI — AYAM
    "ayam"                          : "ayam rata-rata",
    "daging ayam"                   : "daging ayam ayam dan kulit mentah",
    "dada ayam"                     : "daging dada ayam dan kulit mentah",
    "dada ayam fillet"              : "daging dada ayam dan kulit mentah",
    "dada ayam filet"               : "daging dada ayam dan kulit mentah",
    "dada fillet"                   : "daging dada ayam dan kulit mentah",
    "fillet ayam"                   : "daging dada ayam dan kulit mentah",
    "filet ayam"                    : "daging dada ayam dan kulit mentah",
    "fillet dada ayam"              : "daging dada ayam dan kulit mentah",
    "fillèt dada ayam"              : "daging dada ayam dan kulit mentah",
    "dada ayam giling"              : "ayam cincang",
    "daging ayam giling"            : "ayam cincang",
    "ayam giling"                   : "ayam cincang",
    "paha ayam"                     : "daging paha ayam dan kulit mentah",
    "daging paha ayam"              : "daging paha ayam dan kulit mentah",
    "ayam paha"                     : "daging paha ayam dan kulit mentah",
    "ayam paha atas"                : "daging paha ayam dan kulit mentah",
    "ayam paha gede dada ayam"      : "daging paha ayam dan kulit mentah",
    "ayam paha sayap"               : "daging paha ayam dan kulit mentah",
    "sayap ayam"                    : "daging kaki ayam dan kulit mentah",
    "sayap leher ayam suka"         : "daging kaki ayam dan kulit mentah",
    "ayam sayap"                    : "daging kaki ayam dan kulit mentah",
    "sayap ayam drum stick"         : "daging kaki ayam dan kulit mentah",
    "kepala ayam"                   : "ayam rata-rata",
    "leher ayam"                    : "ayam rata-rata",
    "hati ayam"                     : "ayam rata-rata",
    "ati ayam"                      : "ayam rata-rata",
    "ampela ayam"                   : "ayam rata-rata",
    "kulit ayam"                    : "ayam rata-rata",
    "ayam kampung"                  : "ayam rata-rata",
    "ayam broiler"                  : "ayam rata-rata",
    "ayam pejantan"                 : "ayam rata-rata",
    "ayam tulang"                   : "ayam rata-rata",
    "daging tulangan ayam"          : "ayam rata-rata",
    "rollade ayam"                  : "ayam rata-rata",
    "ayam dada"                     : "daging dada ayam dan kulit mentah",
    "ayam filet"                    : "daging dada ayam dan kulit mentah",
    "ayam leher"                    : "ayam rata-rata",
    "tulang ayam punggung"          : "ayam rata-rata",

    # ── PROTEIN HEWANI — SAPI
    "daging sapi"                   : "daging sapi rata-rata",
    "daging"                        : "daging sapi rata-rata",
    "sapi"                          : "daging sapi rata-rata",
    "daging sapi giling"            : "daging sapi cincang 10-15% lemak mentah",
    "daging giling"                 : "daging sapi cincang 10-15% lemak mentah",
    "sapi giling"                   : "daging sapi cincang 10-15% lemak mentah",
    "daging sapu giling"            : "daging sapi cincang 10-15% lemak mentah",
    "iga sapi"                      : "daging sapi rata-rata",
    "iga"                           : "daging sapi rata-rata",
    "daging iga"                    : "daging sapi rata-rata",
    "daging sapi iga"               : "daging sapi rata-rata",
    "daging has"                    : "daging sapi rata-rata",
    "daging sapi has"               : "daging sapi rata-rata",
    "daging sapi khas"              : "daging sapi rata-rata",
    "daging sapi lulur"             : "daging sapi rata-rata",
    "daging sapi sengkel"           : "daging sapi rata-rata",
    "tetelan"                       : "daging sapi rata-rata",
    "tetelan sapi"                  : "daging sapi rata-rata",
    "daging sapi tetelan"           : "daging sapi rata-rata",
    "kikil"                         : "daging sapi rata-rata",
    "kikil sapi"                    : "daging sapi rata-rata",
    "hati sapi"                     : "daging sapi rata-rata",
    "lemak sapi"                    : "daging sapi rata-rata",
    "lidah sapi"                    : "daging sapi rata-rata",
    "tulangan sapi"                 : "daging sapi rata-rata",
    "tenderloin aussie"             : "daging sapi rata-rata",
    "rib eye slice"                 : "daging sapi rata-rata",
    "daging sapi bole"              : "daging sapi rata-rata",
    "danging"                       : "daging sapi rata-rata",

    # ── PROTEIN HEWANI — KAMBING/DOMBA
    "kambing"                       : "nilai rata-rata daging domba mentah",
    "daging kambing"                : "nilai rata-rata daging domba mentah",
    "iga kambing"                   : "nilai rata-rata daging domba mentah",
    "paha kambing muda"             : "kaki domba tidak ditentukan mentah",
    "daging paha kambing muda"      : "kaki domba tidak ditentukan mentah",
    "kepala kambing"                : "nilai rata-rata daging domba mentah",
    "tulang kambing daging kambing" : "nilai rata-rata daging domba mentah",
    "tulangan kambing"              : "nilai rata-rata daging domba mentah",
    "kaki kambing presto empuk"     : "kaki domba tidak ditentukan mentah",
    "jeroan kambing"                : "nilai rata-rata daging domba mentah",

    #  PROTEIN HEWANI — IKAN AIR TAWAR
    "ikan lele"                     : "ikan air tawar mentah",
    "lele"                          : "ikan air tawar mentah",
    "ikan nila"                     : "ikan air tawar mentah",
    "nila"                          : "ikan air tawar mentah",
    "ikan gurami"                   : "ikan air tawar mentah",
    "ikan gurame"                   : "ikan air tawar mentah",
    "gurame"                        : "ikan air tawar mentah",
    "gurami daging tulangnya"       : "ikan air tawar mentah",
    "ikan gurameh"                  : "ikan air tawar mentah",
    "ikan mujair"                   : "ikan air tawar mentah",
    "ikan mujaer"                   : "ikan air tawar mentah",
    "ikan bawal"                    : "ikan air tawar mentah",
    "bawal"                         : "ikan air tawar mentah",
    "ikan patin"                    : "ikan air tawar mentah",
    "ikan gabus"                    : "ikan air tawar mentah",
    "gabus"                         : "ikan air tawar mentah",
    "ikan nila mujair"              : "ikan air tawar mentah",

    #  PROTEIN HEWANI — IKAN LAUT
    "ikan bandeng"                  : "bandeng hering mentah",
    "bandeng"                       : "bandeng hering mentah",
    "ikan kembung"                  : "ikan haring mentah",
    "ikan tongkol"                  : "ikan haring mentah",
    "tongkol"                       : "ikan haring mentah",
    "ikan tenggiri"                 : "ikan haring mentah",
    "daging fillet ikan tenggiri"   : "ikan haring mentah",
    "fillet ikan tenggiri"          : "ikan haring mentah",
    "ikan tuna"                     : "tuna mentah",
    "tuna kalengan"                 : "tuna dalam air kalengan",
    "tuna kemasan"                  : "tuna dalam air kalengan",
    "ikan sarden"                   : "ikan haring mentah",
    "sarden"                        : "ikan haring mentah",
    "ikan pari asap"                : "ikan haring mentah",
    "ikan pindang tongkol"          : "ikan haring mentah",
    "pindang tongkol"               : "ikan haring mentah",
    "ikan dori fillet"              : "ikan trout mentah",
    "dori fillet"                   : "ikan trout mentah",
    "kakap fillet"                  : "ikan trout mentah",
    "kakap merah"                   : "ikan trout mentah",
    "salmon"                        : "budidaya ikan salmon atlantik mentah",
    "salmon fillet"                 : "budidaya ikan salmon atlantik mentah",
    "salmon fillet kulit"           : "budidaya ikan salmon atlantik mentah",
    "ikan fillet mahi mahi"         : "ikan trout mentah",
    "ikan"                          : "ikan haring mentah",
    "ikan teri"                     : "fillet ikan teri dalam minyak zaitun",
    "teri"                          : "fillet ikan teri dalam minyak zaitun",
    "teri nasi"                     : "fillet ikan teri dalam minyak zaitun",
    "teri medan"                    : "fillet ikan teri dalam minyak zaitun",
    "ebi"                           : "budidaya udang harimau raksasa rebus beku",

    #  PROTEIN HEWANI — SEAFOOD
    "udang"                         : "budidaya udang harimau raksasa rebus beku",
    "udang galah"                   : "budidaya udang harimau raksasa rebus beku",
    "udang windu"                   : "budidaya udang harimau raksasa rebus beku",
    "udang tambak"                  : "budidaya udang harimau raksasa rebus beku",
    "udang kepala"                  : "budidaya udang harimau raksasa rebus beku",
    "udang rebon"                   : "budidaya udang harimau raksasa rebus beku",
    "rebon"                         : "budidaya udang harimau raksasa rebus beku",
    "udang pacet"                   : "budidaya udang harimau raksasa rebus beku",
    "cumi"                          : "gurita mentah",
    "cumi cumi"                     : "gurita mentah",
    "kerang"                        : "kerang mentah",
    "kepiting"                      : "kepiting direbus",
    "sotong"                        : "gurita mentah",

    #  PROTEIN HEWANI — TELUR
    "telur"                         : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "telor"                         : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "telur ayam"                    : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "telor ayam"                    : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "telur puyuh"                   : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "putih telur"                   : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "kuning telur"                  : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "kuning telur"                  : "telur ayam kampung ayam kampung (dalam ruangan) mentah",
    "kuning telor"                  : "telur ayam kampung ayam kampung (dalam ruangan) mentah",

    # ── PROTEIN HEWANI — SUSU & TURUNAN
    "susu cair"                     : "susu utuh konvensional (bukan organik) 35 % lemak",
    "susu uht"                      : "susu utuh konvensional (bukan organik) 35 % lemak",
    "susu bubuk"                    : "susu utuh konvensional (bukan organik) 35 % lemak",
    "susu murni hewani"             : "susu utuh konvensional (bukan organik) 35 % lemak",
    "susu sapi murni"               : "susu utuh konvensional (bukan organik) 35 % lemak",
    "yogurt"                        : "yogurt susu murni biasa",
    "yoghurt"                       : "yogurt susu murni biasa",
    "kefir yogurt plain"            : "yogurt susu murni biasa",
    "keju"                          : "keju brie 45% fidm.",
    "keju cheddar"                  : "keju gouda",
    "keju mozarella"                : "keju semihard mozzarella 30% fidm.",
    "keju parmesan"                 : "keju parmesan keras 32 % fidm.",
    "keju quick melt"               : "keju gouda",
    "mentega"                       : "garam mentega ditambahkan",
    "butter"                        : "garam mentega ditambahkan",
    "unsalted butter margarin"      : "garam mentega ditambahkan",
    "salted butter"                 : "garam mentega ditambahkan",
    "margarin"                      : "margarin meja 70% lemak menggunakan lemak nabati lunak",
    "margarine"                     : "margarin meja 70% lemak menggunakan lemak nabati lunak",
    "blue band"                     : "margarin meja 70% lemak menggunakan lemak nabati lunak",
    "cooking cream"                 : "krim berbudaya 18 % lemak",

    # PROTEIN NABATI
    "tahu"                          : "tahu tahu tahu tahu",
    "tahu putih"                    : "tahu tahu tahu tahu",
    "tahu kuning"                   : "tahu tahu tahu tahu",
    "tahu sutera"                   : "tahu tahu tahu tahu",
    "tahu sutra"                    : "tahu tahu tahu tahu",
    "tofu"                          : "tahu tahu tahu tahu",
    "tempe"                         : "bola vegan berbahan dasar kedelai",
    "kacang kedelai"                : "kacang kedelai kering mentah",
    "kedelai"                       : "kacang kedelai kering mentah",
    "kacang hijau"                  : "kacang hijau mentah",
    "kacang merah"                  : "kacang merah",
    "kacang merah basah"            : "kacang merah",
    "kacang hitam"                  : "kacang hitam",
    "kacang polong"                 : "kacang polong hijau mentah",
    "kacang panjang"                : "lilin kacang mentah",
    "kacang pangjang"               : "lilin kacang mentah",
    "kacang tanah"                  : "minyak kacang tanah dipanggang dan diasinkan",
    "kacang mete"                   : "kacang mete panggang kering",
    "kismis"                        : "kismis tanpa biji",
    "kismis hitam"                  : "kismis hitam mentah",
    "raisins"                       : "kismis tanpa biji",
    "tauge"                         : "nilai rata-rata tauge mentah",
    "toge"                          : "nilai rata-rata tauge mentah",
    "taoge"                         : "nilai rata-rata tauge mentah",

    # SAYURAN
    "bayam"                         : "bayam mentah",
    "daun bayam"                    : "bayam mentah",
    "kangkung"                      : "kangkung mentah",
    "sawi"                          : "kubis pak-choi cina mentah",
    "sawi putih"                    : "kubis pak-choi cina mentah",
    "sawi hijau"                    : "kubis pak-choi cina mentah",
    "sawi ijo"                      : "kubis pak-choi cina mentah",
    "sawi caisim"                   : "kubis pak-choi cina mentah",
    "sawi dgn"                      : "kubis pak-choi cina mentah",
    "daun sawi"                     : "kubis pak-choi cina mentah",
    "pakcoy"                        : "kubis pak-choi cina mentah",
    "baby pakcoy"                   : "kubis pak-choi cina mentah",
    "pokcoy"                        : "kubis pak-choi cina mentah",
    "kol"                           : "kubis putih mentah",
    "kubis"                         : "kubis putih mentah",
    "kubis kol"                     : "kubis putih mentah",
    "dun kol"                       : "kubis putih mentah",
    "mangkuk kol"                   : "kubis putih mentah",
    "kembang kol"                   : "kembang kol semua varietas mentah",
    "brokoli"                       : "brokoli mentah",
    "wortel"                        : "wortel mentah",
    "tomat"                         : "tomat matang mentah asal tidak diketahui",
    "tomat merah"                   : "tomat matang mentah asal tidak diketahui",
    "tomat ceri"                    : "tomat matang mentah asal tidak diketahui",
    "timun"                         : "mentimun mentah",
    "mentimun"                      : "mentimun mentah",
    "ketimun"                       : "mentimun ketimun mentah",
    "timun jepang"                  : "mentimun mentah",
    "terong"                        : "terong mentah",
    "labu"                          : "labu mentah",
    "labu siam"                     : "labu mentah",
    "labu baby"                     : "labu mentah",
    "jagung"                        : "biji jagung manis kalengan",
    "jagung pipil"                  : "biji jagung manis kalengan",
    "jagung muda"                   : "bayi jagung",
    "baby corn"                     : "bayi jagung",
    "kentang"                       : "kentang mentah",
    "singkong"                      : "singkong mentah",
    "daun singkong muda"            : "singkong mentah",
    "ubi"                           : "singkong mentah",
    "jamur"                         : "jamur mentah",
    "jamur kancing"                 : "jamur mentah",
    "jamur kuping"                  : "jamur mentah",
    "jamur tiram"                   : "jamur tiram",
    "jamur enoki"                   : "jamur mentah",
    "selada"                        : "kebun selada mentah",
    "daun selada"                   : "selada gunung es (termasuk jenis crisphead) mentah",
    "selada air"                    : "selada gunung es (termasuk jenis crisphead) mentah",
    "buncis"                        : "buncis kalengan",
    "buncis muda"                   : "buncis kalengan",
    "acar timun"                    : "mentimun acar besar",
    "acar"                          : "mentimun acar besar",
    "pare"                          : "terong mentah",
    "lobak"                         : "lobak mentah",
    "bangkuang"                     : "lobak mentah",
    "paprika"                       : "lada manis merah mentah",
    "paprika merah"                 : "lada manis merah mentah",
    "paprika hijau"                 : "lada manis merah mentah",
    "afdhol paprika hijau"          : "lada manis merah mentah",
    "pete"                          : "kacang gula (mangetout kacang salju) mentah",
    "petai"                         : "kacang gula (mangetout kacang salju) mentah",
    "jengkol tua"                   : "kacang gula (mangetout kacang salju) mentah",
    "jengkol"                       : "kacang gula (mangetout kacang salju) mentah",
    "daun pepaya remas"             : "bayam mentah",
    "daun kacang panjang"           : "lilin kacang mentah",
    "daun kedondong"                : "bayam mentah",
    "daun kucai"                    : "daun bawang mentah",
    "daun kuchai"                   : "daun bawang mentah",
    "lokio"                         : "daun bawang mentah",
    "rebung"                        : "rebung kalengan tanpa garam",

    # BUAH
    "jeruk nipis"                   : "jeruk nipis mentah",
    "air jeruk nipis"               : "jeruk nipis mentah",
    "perasan air jeruk nipis"       : "jeruk nipis mentah",
    "jeruk lemon"                   : "lemon mentah",
    "lemon"                         : "lemon mentah",
    "air lemon"                     : "lemon mentah",
    "jeruk"                         : "jeruk mentah",
    "jeruk limau"                   : "jeruk bali mentah",
    "jeruk limo"                    : "jeruk bali mentah",
    "jeruk limao"                   : "jeruk bali mentah",
    "jeruk kesturi"                 : "jeruk bali mentah",
    "jeruk kunci"                   : "jeruk bali mentah",
    "nanas"                         : "nanas mentah",
    "pisang"                        : "pisang mentah",
    "mangga"                        : "mangga mentah",
    "pepaya"                        : "melon mentah",
    "kelapa"                        : "santan",
    "kelapa muda"                   : "santan",
    "air kelapa"                    : "santan",
    "belimbing wuluh"               : "jeruk bali mentah",
    "blimbing wuluh"                : "jeruk bali mentah",
    "belimbing sayur"               : "jeruk bali mentah",
    "asam jawa"                     : "cuka",
    "asam kandis"                   : "cuka",
    "asam sunti"                    : "cuka",
    "asam kesturi"                  : "cuka",
    "asem kandis"                   : "cuka",
    "jus apel"                      : "jus apel",

    #  BUMBU SEGAR
    "bawang putih"                  : "bawang putih mentah",
    "bawang merah"                  : "bawang merah",
    "bawang bombai"                 : "bawang bombay musim semi mentah",
    "bombai"                        : "bawang bombay musim semi mentah",
    "bawang daun"                   : "daun bawang mentah",
    "bawang pre"                    : "daun bawang mentah",
    "bawang prey"                   : "daun bawang mentah",
    "daun bawang"                   : "daun bawang mentah",
    "seledri"                       : "akar seledri seledri mentah",
    "daun seledri"                  : "akar seledri seledri mentah",
    "kemangi"                       : "kemangi segar",
    "daun kemangi"                  : "kemangi segar",

    # ── CABE / CABAI
    # (semua varian → merica cabai pedas mentah)
    "cabe"                          : "merica cabai pedas mentah",
    "cabai"                         : "merica cabai pedas mentah",
    "cabe merah"                    : "merica cabai pedas mentah",
    "cabai merah"                   : "merica cabai pedas mentah",
    "cabe rawit"                    : "merica cabai pedas mentah",
    "cabai rawit"                   : "merica cabai pedas mentah",
    "cabe hijau"                    : "merica cabai pedas mentah",
    "cabai hijau"                   : "merica cabai pedas mentah",
    "cabe keriting"                 : "merica cabai pedas mentah",
    "cabai keriting"                : "merica cabai pedas mentah",
    "cabe ijo"                      : "merica cabai pedas mentah",
    "rawit"                         : "merica cabai pedas mentah",
    "rawit merah"                   : "merica cabai pedas mentah",
    "rawit hijau"                   : "merica cabai pedas mentah",
    "lombok"                        : "merica cabai pedas mentah",
    "chili flakes"                  : "merica cabai pedas mentah",
    "cayenne pepper cabai bubuk"    : "merica cabai pedas mentah",
    "boncabe"                       : "merica cabai pedas mentah",
    "jalapeños"                     : "jalapeños",

    #  REMPAH
    "jahe"                          : "jahe mentah",
    "kunyit"                        : "bubuk kari",
    "kunir"                         : "bubuk kari",
    "lengkuas"                      : "akar peterseli mentah",
    "laos"                          : "akar peterseli mentah",
    "laja"                          : "akar peterseli mentah",
    "langkoas"                      : "akar peterseli mentah",
    "lengkoas"                      : "akar peterseli mentah",
    "serai"                         : "daun bawang mentah",
    "serei"                         : "daun bawang mentah",
    "daun salam"                    : "kemangi segar",
    "daun jeruk"                    : "kemangi segar",
    "daun kunyit"                   : "kemangi segar",
    "daun pandan"                   : "kemangi segar",
    "daun kari"                     : "kemangi segar",
    "kencur"                        : "akar jahe mentah",
    "temu kunci"                    : "akar jahe mentah",
    "kemiri"                        : "kemiri dikeringkan",
    "ketumbar"                      : "bubuk kari",
    "tumbar"                        : "bubuk kari",
    "kluwek"                        : "kastanye mentah",
    "keluwek"                       : "kastanye mentah",

    #  REMPAH KERING
    "lada"                          : "lada hitam",
    "merica"                        : "lada hitam",
    "lada hitam"                    : "lada hitam",
    "lada putih"                    : "lada hitam",
    "merica bubuk"                  : "lada hitam",
    "merica butiran"                : "lada hitam",
    "bubuk kari"                    : "bubuk kari",
    "kari bubuk"                    : "bubuk kari",
    "garam masala"                  : "bubuk kari",
    "garamasala"                    : "bubuk kari",
    "wijen"                         : "biji wijen dihias",
    "oregano"                       : "kemangi dikeringkan",
    "dry oregano"                   : "kemangi dikeringkan",
    "parsley"                       : "peterseli mentah",
    "daun cilantro"                 : "peterseli mentah",
    "italian herbs"                 : "kemangi dikeringkan",
    "saffron"                       : "bubuk kari",
    "ngo hiong"                     : "bubuk kari",
    "ngo hiang"                     : "bubuk kari",

    # BUMBU CAIR / SAUS
    "kecap"                         : "kecap",
    "kecap bango"                   : "kecap",
    "kecap inggris"                 : "kecap",
    "kecap ikan"                    : "kecap",
    "fish sauce"                    : "kecap",
    "kecap shoyu"                   : "kecap",
    "dark soy sauce"                : "kecap",
    "tauco"                         : "kecap",
    "saus tiram"                    : "saus pasta",
    "saos tiram"                    : "saus pasta",
    "saori saos tiram"              : "saus pasta",
    "saori"                         : "saus pasta",
    "saus tomat"                    : "botol saus tomat",
    "saos tomat"                    : "botol saus tomat",
    "pasta tomat"                   : "pasta tomat pekat",
    "saus barbeque"                 : "saus barbeque",
    "saus bbq merk"                 : "saus barbeque",
    "saus blackpepper"              : "saus pasta",
    "saos lada hitam"               : "saus pasta",
    "sauce black pepper"            : "saus pasta",
    "blackpepper"                   : "lada hitam",
    "blackpepper powder"            : "lada hitam",
    "saus cabai"                    : "saus sambal",
    "saus cabe"                     : "saus sambal",
    "petis"                         : "saus sambal",
    "petis udang kualitas super"    : "saus sambal",
    "terasi"                        : "saus sambal",
    "trasi"                         : "saus sambal",
    "mayones"                       : "mayones",
    "mayonnaise"                    : "mayones",
    "mayo"                          : "mayones",
    "mayonaise"                     : "mayones",
    "mirin"                         : "kecap",
    "sake"                          : "kecap",
    "cuka"                          : "cuka",
    "cuka apel"                     : "cuka",
    "cuka makanan"                  : "cuka",
    "pesto"                         : "pesto",

    #  MINYAK
    "minyak goreng"                 : "minyak bunga matahari",
    "minyak sayur"                  : "minyak bunga matahari",
    "minyak"                        : "minyak bunga matahari",
    "minyak wijen"                  : "minyak zaitun",
    "olive oil"                     : "minyak zaitun",
    "minyak zaitun"                 : "minyak zaitun",
    "minyak samin"                  : "minyak bunga matahari",
    "vco"                           : "minyak bunga matahari",

    # ── SANTAN ────────────────────────────────────────────────────
    "santan"                        : "santan",
    "santan kara"                   : "santan",
    "santan kental"                 : "santan",
    "santan instan"                 : "santan",
    "santan bubuk merk sasa plus air": "santan",
    "mangkuk santan kental"         : "santan",

    # ── BUMBU KERING / PENYEDAP ───────────────────────────────────
    "garam"                         : "meja garam",
    "gula pasir"                    : "gula sukrosa putih",
    "gula putih"                    : "gula sukrosa putih",
    "gula merah"                    : "gula merah",
    "gula jawa"                     : "gula merah",
    "gula aren"                     : "gula merah",
    "gula palem"                    : "gula merah",
    "madu"                          : "sayang",
    "minyak ikan"                   : "fillet ikan teri dalam minyak zaitun",

    #  KARBOHIDRAT
    "beras"                         : "nasi setengah matang mentah",
    "nasi"                          : "nasi setengah matang mentah",
    "lontong"                       : "nasi setengah matang mentah",
    "tepung terigu"                 : "tepung gandum utuh",
    "terigu"                        : "tepung gandum utuh",
    "tepung beras"                  : "tepung beras",
    "tepung maizena"                : "tepung jagung",
    "maizena"                       : "tepung jagung",
    "tepung tapioka"                : "tepung kentang",
    "tapioka"                       : "tepung kentang",
    "tepung sagu"                   : "tepung kentang",
    "sagu tani"                     : "tepung kentang",
    "tepung roti"                   : "tepung roti",
    "tepung panir"                  : "tepung roti",
    "bread crumb"                   : "tepung roti",
    "breadcrumb"                    : "tepung roti",
    "bread crumbs"                  : "tepung roti",
    "mie"                           : "bihun",
    "mi"                            : "bihun",
    "bihun"                         : "bihun",
    "sohun"                         : "bihun",
    "soun"                          : "bihun",
    "pasta"                         : "pasta mentah",
    "spaghetti"                     : "pasta mentah",
    "spagheti"                      : "pasta mentah",
    "spageti"                       : "pasta mentah",
    "makaroni"                      : "pasta mentah",
    "macaroni elbow"                : "pasta mentah",
    "macarone"                      : "pasta mentah",
    "roti"                          : "roti burger",
    "roti tawar"                    : "roti putih gulung butiran kasar",
    "roti kebab"                    : "roti tortilla gandum",
    "kulit pangsit"                 : "nilai rata-rata pangsit",
    "kulit lumpia"                  : "nilai rata-rata pangsit",
    "puff pastry"                   : "nilai rata-rata pangsit",
    "pasta lasagna"                 : "pasta mentah",
    "gandum"                        : "tepung gandum utuh",

    #  OLAHAN
    "bakso"                         : "bakso",
    "bakso sapi"                    : "bakso",
    "bakso ikan"                    : "bakso ikan kalengan",
    "sosis"                         : "sosis ayam",
    "sosis sapi"                    : "sosis ayam",
    "smoked beef"                   : "bacon digoreng mentah",
    "kornet sapi"                   : "ham putih",
    "emping"                        : "keripik kentang",
    "kerupuk"                       : "keripik kentang",
    "kerupuk udang"                 : "keripik kentang",
    "nori"                          : "kemangi dikeringkan",
}

### 4. normalisasi nama_id untuk matching

In [ ]:
def norm_nama(text: str) -> str:
    return str(text).lower().strip()

### 5. token match — hanya untuk pola yang aman dan pasti

In [ ]:
TOKEN_RULES = [
    # Daging sapi — semua variant mengandung "sapi"
    ("sapi",        "daging sapi rata-rata"),
    # Daging ayam — variant dengan "ayam" yang belum di manual map
    ("ayam",        "daging ayam ayam dan kulit mentah"),
    # Kambing
    ("kambing",     "nilai rata-rata daging domba mentah"),
    # Udang
    ("udang",       "budidaya udang harimau raksasa rebus beku"),
    # Ikan air tawar generik
    ("gurame",      "ikan air tawar mentah"),
    ("gurami",      "ikan air tawar mentah"),
    ("lele",        "ikan air tawar mentah"),
    ("nila",        "ikan air tawar mentah"),
    # Seafood
    ("cumi",        "gurita mentah"),
    # Telur
    ("telur",       "telur ayam kampung ayam kampung (dalam ruangan) mentah"),
    ("telor",       "telur ayam kampung ayam kampung (dalam ruangan) mentah"),
    # Susu
    ("susu",        "susu utuh konvensional (bukan organik) 35 % lemak"),
    # Tahu
    ("tahu",        "tahu tahu tahu tahu"),
    # Tempe
    ("tempe",       "bola vegan berbahan dasar kedelai"),
    # Sayuran
    ("bayam",       "bayam mentah"),
    ("kangkung",    "kangkung mentah"),
    ("brokoli",     "brokoli mentah"),
    ("wortel",      "wortel mentah"),
    ("tomat",       "tomat matang mentah asal tidak diketahui"),
    ("jagung",      "biji jagung manis kalengan"),
    ("kentang",     "kentang mentah"),
    ("jamur",       "jamur mentah"),
    ("terong",      "terong mentah"),
    ("labu",        "labu mentah"),
    ("singkong",    "singkong mentah"),
    ("selada",      "kebun selada mentah"),
    ("buncis",      "buncis kalengan"),
    ("paprika",     "lada manis merah mentah"),
    # Bumbu
    ("bawang putih","bawang putih mentah"),
    ("bawang merah","bawang merah"),
    ("jahe",        "akar jahe mentah"),
    ("kunyit",      "bubuk kari"),
    ("kemiri",      "kemiri dikeringkan"),
    ("santan",      "santan"),
    ("kecap",       "kecap"),
    ("madu",        "sayang"),
    # Buah
    ("jeruk nipis", "jeruk nipis mentah"),
    ("pisang",      "pisang mentah"),
    ("mangga",      "mangga mentah"),
    ("nanas",       "nanas mentah"),
    # Karbohidrat
    ("beras",       "nasi setengah matang mentah"),
    ("tepung terigu","tepung gandum utuh"),
    ("tepung beras","tepung beras"),
    ("tepung maizena","tepung jagung"),
    ("maizena",     "tepung jagung"),
    ("tepung tapioka","tepung kentang"),
    ("tapioka",     "tepung kentang"),
    ("mie",         "bihun"),
    ("pasta",       "pasta mentah"),
    # Protein nabati
    ("kacang hijau","kacang hijau mentah"),
    ("kacang merah","kacang merah"),
    ("kacang panjang","lilin kacang mentah"),
    ("kacang tanah","minyak kacang tanah dipanggang dan diasinkan"),
    ("kacang kedelai","kacang kedelai kering mentah"),
    # Olahan
    ("bakso",       "bakso"),
    ("sosis",       "sosis ayam"),
]

def token_match(nama_norm: str) -> tuple | None:
    """Cek apakah nama mengandung salah satu token rule secara aman."""
    for keyword, carbon_key in TOKEN_RULES:
        if keyword in nama_norm:
            if carbon_key in carbon:
                return (carbon[carbon_key][1], f"token:{keyword}", carbon_key)
    return None

### 6. lapisan matching

In [ ]:
FUZZY_THRESHOLD = 85

def get_co2e(nama_id: str) -> tuple:
    """
    Return: (co2e | None, match_type, matched_key)
    match_type: 'manual' | 'exact' | 'token:...' | 'fuzzy:xx%' | 'no_match'
    """
    norm = norm_nama(nama_id)

    # Pra-Layer: Manual map (presisi tertinggi)
    if norm in MANUAL_MAP:
        target = MANUAL_MAP[norm]
        if target in carbon:
            return (carbon[target][1], "manual", target)

    # Layer 1: Exact match (setelah lowercase + strip)
    if norm in carbon:
        return (carbon[norm][1], "exact", norm)

    # Layer 2: Token match (aturan eksplisit, bukan bebas)
    result = token_match(norm)
    if result:
        return result

    # Layer 3: Fuzzy match
    fuzzy_result = process.extractOne(
        norm,
        carbon_keys,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_THRESHOLD,
    )
    if fuzzy_result:
        matched_key, score, _ = fuzzy_result
        return (carbon[matched_key][1], f"fuzzy:{score:.0f}%", matched_key)

    return (None, "no_match", "")

### 7. proses semua bahan

In [ ]:
print("\n  Menjalankan 3-layer matching...\n")

co2e_vals, match_types, match_keys = [], [], []
cnt = {"manual": 0, "exact": 0, "token": 0, "fuzzy": 0, "no_match": 0}

for _, row in df.iterrows():
    co2e, mtype, mkey = get_co2e(row["nama_id"])
    co2e_vals.append(co2e)
    match_types.append(mtype)
    match_keys.append(mkey)
    if   mtype == "manual":           cnt["manual"]   += 1
    elif mtype == "exact":            cnt["exact"]    += 1
    elif mtype.startswith("token"):   cnt["token"]    += 1
    elif mtype.startswith("fuzzy"):   cnt["fuzzy"]    += 1
    else:                             cnt["no_match"] += 1

df["karbon_co2e"]  = co2e_vals
df["_match_type"]  = match_types
df["_matched_key"] = match_keys

### 8. statistik

In [ ]:
total   = len(df)
matched = total - cnt["no_match"]

print("═" * 62)
print("  LAPORAN HASIL CARBON MERGE")
print("═" * 62)
print(f"  Total bahan diproses         : {total}")
print(f"  {'─'*44}")
print(f"  Pra-Layer  Manual map        : {cnt['manual']:>5}  ({cnt['manual']/total*100:.1f}%)")
print(f"  Layer 1    Exact match       : {cnt['exact']:>5}  ({cnt['exact']/total*100:.1f}%)")
print(f"  Layer 2    Token match       : {cnt['token']:>5}  ({cnt['token']/total*100:.1f}%)")
print(f"  Layer 3    Fuzzy match       : {cnt['fuzzy']:>5}  ({cnt['fuzzy']/total*100:.1f}%)")
print(f"  {'─'*44}")
print(f"   Total matched (ada CO₂e)  : {matched:>5}  ({matched/total*100:.1f}%)")
print(f"   No match (→ NaN)          : {cnt['no_match']:>5}  ({cnt['no_match']/total*100:.1f}%)")
print("═" * 62)

# Rata-rata CO₂e per kategori
print("\n🌿 RATA-RATA CO₂e PER KATEGORI (kg CO₂e / kg bahan):")
print(f"  {'Kategori':<22} {'n':>5}  {'Avg':>8}  {'Max':>9}")
print(f"  {'─'*48}")
cat_stats = (
    df[df["karbon_co2e"].notna()]
    .groupby("kategori")
    .agg(n=("karbon_co2e", "count"),
         avg=("karbon_co2e", "mean"),
         max_=("karbon_co2e", "max"))
    .round(3).sort_values("avg", ascending=False)
)
for cat, r in cat_stats.iterrows():
    print(f"  {cat:<22} {int(r['n']):>5}  {r['avg']:>8.3f}  {r['max_']:>9.3f}")

# Top-10 bahan berdampak tertinggi
print("\n TOP-10 BAHAN BERDAMPAK KARBON TERTINGGI:")
top10 = (df[df["karbon_co2e"].notna()]
         .nlargest(10, "karbon_co2e")
         [["nama_id", "kategori", "frekuensi", "karbon_co2e"]])
print(top10.to_string(index=False))

# Bahan yang masih NaN
no_co2e = df[df["karbon_co2e"].isna()].sort_values("frekuensi", ascending=False)
print(f"\n TOP-20 BAHAN TANPA CO₂e (tetap NaN — no fallback):")
print(f"  {'nama_id':<40}  {'frekuensi':>10}  kategori")
print(f"  {'─'*65}")
for _, r in no_co2e.head(20).iterrows():
    print(f"  {r['nama_id']:<40}  {r['frekuensi']:>10}  {r['kategori']}")


FINAL_COLS = [
    "nama_id", "frekuensi", "kategori",
    "umur_kulkas", "umur_suhu_ruang", "umur_freezer",
    "kalori_per_100g", "protein_g", "lemak_g", "karbo_g",
    "karbon_co2e",
]
df_final = df[FINAL_COLS].copy()
df_final["karbon_co2e"] = pd.to_numeric(df_final["karbon_co2e"], errors="coerce").round(3)

### 9. export CVS dan JSON

In [ ]:
# Export CSV
df_final.to_csv("ingredients_master_final.csv", index=False)

# Export JSON — NaN → null (None) agar valid JSON
records = []
for _, row in df_final.iterrows():
    rec = row.to_dict()
    for k, v in rec.items():
        if isinstance(v, float) and np.isnan(v):
            rec[k] = None
    records.append(rec)

with open("ingredients_master_final.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)


print(f"   ingredients_master_final.csv")
print(f"     {len(df_final)} baris  ×  {len(FINAL_COLS)} kolom")
print(f"   ingredients_master_final.json")
print(f"     {len(records)} objek  |  orient='records'  |  NaN → null ✓")
print("═" * 62)
